In [124]:
!pip install pandas deep-translator

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [125]:
pip install googletrans==4.0.0-rc1

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [126]:
import pandas as pd

csv_path = r"C:\xampp\htdocs\TNL6323-Project\google_maps_reviews_Ichi Zen Melaka.csv"

df = pd.read_csv(csv_path)

print(f"Total reviews: {len(df)}")
df.head()

Total reviews: 200


,name,author,rating,text,date
0,reeves ngim,NaN,3,NaN,2026-06-13T08:54:59.132Z
1,Gan Ryan,NaN,5,NaN,2026-06-07T05:56:31.538Z
2,Rita Fam,NaN,5,Food delicious and impressed with the staff se...,2026-06-04T10:42:46.129Z
3,Flower Soul,NaN,5,"Food tasted good, menu have alot of varieties ...",2026-05-30T11:35:50.187Z
4,FLY4SIX9,NaN,5,Taste good & nice service! Will come again,2026-05-15T11:11:43.061Z


In [127]:
# Remove empty, null, or whitespace-only reviews
df = df[df["text"].notna()]          # remove NaN
df = df[df["text"].astype(str).str.strip() != ""]  # remove empty strings

# Reset index after filtering
df = df.reset_index(drop=True)

print(f"Remaining reviews: {len(df)}")
df.head()

Remaining reviews: 132


,name,author,rating,text,date
0,Rita Fam,NaN,5,Food delicious and impressed with the staff se...,2026-06-04T10:42:46.129Z
1,Flower Soul,NaN,5,"Food tasted good, menu have alot of varieties ...",2026-05-30T11:35:50.187Z
2,FLY4SIX9,NaN,5,Taste good & nice service! Will come again,2026-05-15T11:11:43.061Z
3,S Chin,NaN,5,Good food good service,2026-05-15T10:50:28.908Z
4,Kenny Khoo,NaN,5,Went here to celebrate birthday and their food...,2026-05-13T01:39:32.529Z


In [128]:
reviews = df["text"].fillna("").astype(str)

print(reviews.iloc[0])

Food delicious and impressed with the staff service. They go all the way out to help customer needs. A place where I will definitely visit again


In [129]:
from googletrans import Translator
import pandas as pd
import time

translator = Translator()

def translate_long_text_googletrans(text, chunk_size=4000):
    if pd.isna(text) or not str(text).strip():
        return ""
    
    text = str(text).strip()
    
    # Split into chunks (sentences better, but char-based works)
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    translated_chunks = []
    
    for chunk in chunks:
        for attempt in range(3):  # retry up to 3 times
            try:
                # Detect source language (optional: force 'zh' for Chinese)
                translated = translator.translate(chunk, dest='en', src='auto')
                translated_chunks.append(translated.text)
                time.sleep(0.3)
                break
            except Exception as e:
                print(f"Attempt {attempt+1} failed: {e}")
                time.sleep(1)
        else:
            # If all retries fail, keep original chunk
            translated_chunks.append(chunk)
    
    return " ".join(translated_chunks)

# Apply to your dataframe
df["translated_text"] = df["text"].apply(translate_long_text_googletrans)

In [130]:
# ---------- SECTION: Arrange & Rename the 'number' column ----------

# Replace the 'number' column with a new sequential order (starting from 1)
df['number'] = range(1, len(df) + 1)

# 4. Display the first few rows to verify
print(df[['number', 'name', 'rating', 'translated_text']].head())

   number         name  rating  \
0       1     Rita Fam       5   
1       2  Flower Soul       5   
2       3     FLY4SIX9       5   
3       4       S Chin       5   
4       5   Kenny Khoo       5   

                                     translated_text  
0  Food delicious and impressed with the staff se...  
1  Food tasted good, menu have alot of varieties ...  
2         Taste good & nice service! Will come again  
3                             Good food good service  
4  Went here to celebrate birthday and their food...  


In [131]:
display(
    df[
        [
            "number",
            "name",
            "rating",
            #"text",
            "translated_text"
        ]
    ].head(20)
)

,number,name,rating,translated_text
0,1,Rita Fam,5,Food delicious and impressed with the staff se...
1,2,Flower Soul,5,"Food tasted good, menu have alot of varieties ..."
2,3,FLY4SIX9,5,Taste good & nice service! Will come again
3,4,S Chin,5,Good food good service
4,5,Kenny Khoo,5,Went here to celebrate birthday and their food...
5,6,Susan Yong,4,Good food and clean environment. Prompt and po...
6,7,Allan Pang,5,Great place for Japanese food in Melaka.
7,8,Kai Chee Too,5,The Best Japanese restaurant in Melaka. Served...
8,9,Vangilyn Lim,4,"The food was good, staff were very nice, also ..."
9,10,DERRYCK KOO,3,"Above average ""serve all"" type of Japanese res..."


In [132]:
output_path = r"C:\xampp\htdocs\TNL6323-Project\google_maps_reviews_Ichi Zen Melaka_translated.csv"

df_to_save = df[[
    "number",
    "name",
    "rating",
    #"text",
    "translated_text"
]]

df_to_save.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved to: {output_path}")

Saved to: C:\xampp\htdocs\TNL6323-Project\google_maps_reviews_Ichi Zen Melaka_translated.csv


In [133]:
for i, text in enumerate(df["translated_text"], start=1):
    print(f"{i}. {text}\n")

1. Food delicious and impressed with the staff service. They go all the way out to help customer needs. A place where I will definitely visit again

2. Food tasted good, menu have alot of varieties that other places dont have, price quite reasonable, staffs also very attentive and friendly

3. Taste good & nice service! Will come again

4. Good food good service

5. Went here to celebrate birthday and their food and service was top notch. The food we ordered came fast even with the high volume of customers they had.

Everything was perfect. The food was delicious and the portion was big, their service was great as they gave high attention to their customers.

For birthdays, you get a selection of their free dish and that too was nice. For each RM50 spent, they will treat you to a free scoop of ice cream of the day. The outlet manager was so nice that he allowed us to trade one ice cream to a matcha ice cream.

They had dining time limit when we went. You can only spend 1.5 hours but th